# Train/Test Split Notebook (Noob‑Friendly)

This notebook takes your **canonical universal JSONL** file and creates a **fixed train/test split**.

**You run this once** per split version. The test set should stay the same forever so your accuracies are comparable.


## 0. Settings
Edit these paths/values once and re‑run the notebook top‑to‑bottom.

In [1]:
from pathlib import Path
import json, random
from datetime import datetime

# ------------------------------------------------------------
# Locate repo root (same logic as XLSX → JSONL notebook)
# ------------------------------------------------------------
def find_repo_root(start: Path | None = None) -> Path:
    start = Path.cwd() if start is None else Path(start).resolve()
    for p in [start] + list(start.parents):
        if (p / ".git").exists() or (p / "README.md").exists():
            return p
    raise RuntimeError(f"Repo root not found from start={start}")

repo_root = find_repo_root()
print("Repo root:", repo_root)

# ------------------------------------------------------------
# Path to canonical JSONL (universal dataset)
# Matches your XLSX → JSONL OUT_PATH logic
# ------------------------------------------------------------
INPUT_JSONL = (
    repo_root
    / "datasets"
    / "yt_factlink"
    / "01_conversion"
    / "02_outputs"
    / "manual_labels_386_v2.data.jsonl"
)

# ------------------------------------------------------------
# Where to write the split (versioned split folder)
# ------------------------------------------------------------
SPLIT_DIR = (
    repo_root
    / "datasets"
    / "yt_factlink"
    / "01_conversion"
    / "03_splits"
    / "split_v1_seed42"
)

# ------------------------------------------------------------
# Split settings
# ------------------------------------------------------------
TRAIN_RATIO = 0.8   # 80% train, 20% test
SEED = 42           # fixed seed => deterministic split

SPLIT_DIR.mkdir(parents=True, exist_ok=True)

print("Input JSONL:", INPUT_JSONL.resolve())
print("Output split dir:", SPLIT_DIR.resolve())



Repo root: /Users/simonhinterreiter/Library/CloudStorage/Dropbox/06 - Coding/01 - Local Git Repos/02 - Mac/ici01_PublicOpinionAnalyzer
Input JSONL: /Users/simonhinterreiter/Library/CloudStorage/Dropbox/06 - Coding/01 - Local Git Repos/02 - Mac/ici01_PublicOpinionAnalyzer/datasets/yt_factlink/01_conversion/02_outputs/manual_labels_386_v2.data.jsonl
Output split dir: /Users/simonhinterreiter/Library/CloudStorage/Dropbox/06 - Coding/01 - Local Git Repos/02 - Mac/ici01_PublicOpinionAnalyzer/datasets/yt_factlink/01_conversion/03_splits/split_v1_seed42


## 1. Load canonical JSONL
Each line is one JSON object.

In [2]:
def read_jsonl(path: Path):
    data = []
    with path.open("r", encoding="utf-8") as f:
        for i, line in enumerate(f):
            line = line.strip()
            if not line:
                continue
            obj = json.loads(line)
            data.append(obj)
    return data

data = read_jsonl(INPUT_JSONL)
print("Loaded examples:", len(data))
data[0]


Loaded examples: 386


{'id': 'syn_0000',
 'VIDEOTITEL': 'The Chinese representative angrily said "Taiwan belongs to China"!',
 'VIDEOCONTEXT': "At the 78th Special Session of the UN General Assembly, China's representative forcefully asserted the one-China principle, claiming Taiwan as inalienable sovereign territory and warning against independence. Japan's representative, Ono Kosei, countered calmly by asking a profound question: if Taiwan truly belongs to China, why is military force necessary for unification? He urged the assembly to respect the principle of self-determination and the wishes of the 23 million Taiwanese people. This thoughtful challenge disrupted the aggressive momentum, sparking a moment of silence and later earning support from U.S. and European delegates. Following the public impasse, the Chinese and Japanese diplomats shared a rare private moment, candidly discussing their difficult positions and expressing a mutual hope for future peace. The entire incident rekindled global debate o

## 2. Basic sanity checks
We verify that every example has an `id` field.

You said the IDs look like `syn_0000`, `syn_0001`, ...

In [3]:
missing_id = [i for i, ex in enumerate(data) if "id" not in ex or not ex["id"]]
if missing_id:
    raise ValueError(f"Found {len(missing_id)} examples without id. First few: {missing_id[:5]}")

print("All examples have an id ✅")


All examples have an id ✅


## 3. Deterministic shuffle + split

- We shuffle using a fixed seed.
- Then we slice into train/test.
- Same seed + same input file => same split forever.

In [4]:
rng = random.Random(SEED)
shuffled = data[:]  # copy
rng.shuffle(shuffled)

n_total = len(shuffled)
n_train = int(round(n_total * TRAIN_RATIO))
n_train = max(1, min(n_train, n_total - 1))  # safety for tiny sets

train = shuffled[:n_train]
test  = shuffled[n_train:]

print("Total:", n_total)
print("Train:", len(train))
print("Test :", len(test))


Total: 386
Train: 309
Test : 77


## 4. Save train/test JSONL

In [5]:
def write_jsonl(path: Path, rows):
    with path.open("w", encoding="utf-8") as f:
        for r in rows:
            f.write(json.dumps(r, ensure_ascii=False) + "\n")

train_path = SPLIT_DIR / "train.data.jsonl"
test_path  = SPLIT_DIR / "test.data.jsonl"

write_jsonl(train_path, train)
write_jsonl(test_path, test)

print("Wrote:")
print(" -", train_path.resolve())
print(" -", test_path.resolve())


Wrote:
 - /Users/simonhinterreiter/Library/CloudStorage/Dropbox/06 - Coding/01 - Local Git Repos/02 - Mac/ici01_PublicOpinionAnalyzer/datasets/yt_factlink/01_conversion/03_splits/split_v1_seed42/train.data.jsonl
 - /Users/simonhinterreiter/Library/CloudStorage/Dropbox/06 - Coding/01 - Local Git Repos/02 - Mac/ici01_PublicOpinionAnalyzer/datasets/yt_factlink/01_conversion/03_splits/split_v1_seed42/test.data.jsonl


## 5. Write split manifest
This keeps the split reproducible + comparable.

In [6]:
manifest = {
    "created_at_utc": datetime.utcnow().isoformat() + "Z",
    "input_file": str(INPUT_JSONL.resolve()),
    "seed": SEED,
    "train_ratio": TRAIN_RATIO,
    "counts": {
        "total": n_total,
        "train": len(train),
        "test": len(test),
    },
    "train_ids": [ex["id"] for ex in train],
    "test_ids": [ex["id"] for ex in test],
    "notes": "Fixed split. Do not overwrite; create a new split folder if needed."
}

manifest_path = SPLIT_DIR / "split_manifest.json"
manifest_path.write_text(json.dumps(manifest, ensure_ascii=False, indent=2), encoding="utf-8")

print("Manifest written to:", manifest_path.resolve())


Manifest written to: /Users/simonhinterreiter/Library/CloudStorage/Dropbox/06 - Coding/01 - Local Git Repos/02 - Mac/ici01_PublicOpinionAnalyzer/datasets/yt_factlink/01_conversion/03_splits/split_v1_seed42/split_manifest.json


/var/folders/98/dyrftdkj6hl9_qxt7gl5_vjr0000gn/T/ipykernel_48399/2185445420.py:2: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  "created_at_utc": datetime.utcnow().isoformat() + "Z",


## Done
You now have a fixed split:

- `train.data.jsonl` -> used for fine‑tuning
- `test.data.jsonl` -> used for accuracy evaluation only

**Never train on the test set.**